# 🎨 AbiArt — Kaggle GPU Backend (Automatic1111 SD WebUI API)

Ce notebook transforme votre environnement **Kaggle GPU gratuit (30h/semaine)** en serveur pour votre studio **AbiArt**.

### ⚡ Configuration requise dans Kaggle (panneau à droite) :
1. **Accelerator** : Choisir `GPU T4 x2` (ou `P100`)
2. **Internet** : Activer `Internet ON`
3. Cliquez sur **Run All** (ou exécutez les cellules dans l'ordre).
4. À la dernière cellule, copiez l'URL générée (`https://xxxx.trycloudflare.com`) et collez-la dans AbiArt !

In [ ]:
# 1. Vérification du GPU
!nvidia-smi

In [ ]:
# 2. Résolution des dépendances (CLIP / setuptools / open_clip / Pillow moderne)
# Note: pip moderne sous Python 3.11/3.12 n'a pas de wheel pour les vieux paquets comme Pillow 9.5 ou CLIP.
# On installe les versions précompilées modernes et sans isolation de build.

!pip install -q "setuptools<70" wheel ftfy regex tqdm "Pillow>=10.0.0"
!pip install -q --no-build-isolation https://github.com/openai/CLIP/archive/d50d76daa670286dd6cacf3bcd80b5e4823fc8e1.zip
!pip install -q --no-build-isolation https://github.com/mlfoundations/open_clip/archive/bb44a442de3a22444740f64bba978707a0068305.zip
!pip install -q xformers ultralytics mediapipe pyngrok

print("✓ CLIP, Pillow et dépendances installés avec succès !")

In [ ]:
# 3. Installation d'Automatic1111 WebUI et patch des versions incompatibles
import os
%cd /kaggle/working

# Cloner SD WebUI si non présent
if not os.path.exists('/kaggle/working/stable-diffusion-webui'):
    !git clone https://github.com/AUTOMATIC1111/stable-diffusion-webui.git /kaggle/working/stable-diffusion-webui

%cd /kaggle/working/stable-diffusion-webui

# Fix 1 : Le dépôt Stability-AI original est privé. On définit le miroir public officiel :
os.environ["STABLE_DIFFUSION_REPO"] = "https://github.com/w-e-w/stablediffusion.git"

# Fix 2 : Remplacer l'ancienne version Pillow==9.5.0 qui ne compile pas sous Python 3.11/3.12 par Pillow>=10.0.0
!sed -i 's/Pillow==9.5.0/Pillow>=10.0.0/g' requirements_versions.txt 2>/dev/null || true
!sed -i 's/Pillow==9.5.0/Pillow>=10.0.0/g' requirements.txt 2>/dev/null || true

# Extensions populaires (ADetailer pour corriger visages et mains, ControlNet)
!git clone --depth 1 https://github.com/Bing-su/adetailer.git extensions/adetailer 2>/dev/null || true
!git clone --depth 1 https://github.com/Mikubill/sd-webui-controlnet.git extensions/sd-webui-controlnet 2>/dev/null || true

print("✓ Automatic1111 configuré et patché avec succès !")

In [ ]:
# 4. Téléchargement de modèles par défaut (Animagine XL 3.1 & SDXL VAE)
%cd /kaggle/working/stable-diffusion-webui/models/Stable-diffusion

if not os.path.exists('animagine-xl-3.1.safetensors'):
    print("Téléchargement d'Animagine XL 3.1...")
    !wget -q -c https://huggingface.co/cagliostrolab/animagine-xl-3.1/resolve/main/animagine-xl-3.1.safetensors -O animagine-xl-3.1.safetensors

%cd /kaggle/working/stable-diffusion-webui/models/VAE
if not os.path.exists('sdxl_vae.safetensors'):
    !wget -q -c https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors -O sdxl_vae.safetensors

%cd /kaggle/working/stable-diffusion-webui
print("✓ Modèles prêts !")

In [ ]:
# 5. Installation de Cloudflare Tunnel (Tunnel gratuit sans compte)
!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1
print("✓ Cloudflared prêt !")

In [ ]:
# 6. Démarrage du Tunnel et de l'API Automatic1111
import os
import subprocess
import time
import re
import threading

os.environ["STABLE_DIFFUSION_REPO"] = "https://github.com/w-e-w/stablediffusion.git"

# Démarrer le tunnel Cloudflare en tâche de fond
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

def print_tunnel_url():
    for line in iter(tunnel_proc.stderr.readline, ''):
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            print("\n" + "="*60)
            print("🎨 URL KAGGLE POUR ABIART :")
            print(f"   >>>  {url}  <<<")
            print("="*60)
            print("Copiez cette URL et collez-la dans AbiArt → Backend: Kaggle → Connect\n")
            break

threading.Thread(target=print_tunnel_url, daemon=True).start()
time.sleep(2)

# Lancement de l'API Automatic1111
!python launch.py \
    --enable-insecure-extension-access \
    --skip-torch-cuda-test \
    --no-half-vae \
    --xformers \
    --api \
    --listen \
    --port 7860